# FakeNewsNet Dataset — Exploratory Data Analysis

FakeNewsNet is a binary fake/real news classification dataset.  
This Kaggle version provides flat CSVs with article title, body text, and metadata.

**Files available:**
- `PolitiFact_fake_news_content.csv` / `PolitiFact_real_news_content.csv` — expert-verified political news
- `BuzzFeed_fake_news_content.csv` / `BuzzFeed_real_news_content.csv` — BuzzFeed political news

**Note:** This version does NOT have the `gossipcop` sub-dataset or the directory tree with `news_content.json` files. The benchmark loader will be updated to read these CSVs directly.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = Path('../datasets/FakeNewsNet')

def load_fnn(source: str) -> pd.DataFrame:
    """Load fake + real CSVs for a source, attach label column."""
    dfs = []
    for label in ('fake', 'real'):
        path = DATA_DIR / f'{source}_{label}_news_content.csv'
        if not path.exists():
            print(f'  Missing: {path.name}')
            continue
        df = pd.read_csv(path, low_memory=False)
        df['label'] = label
        df['binary'] = 1 if label == 'fake' else 0
        df['source'] = source
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

pf = load_fnn('PolitiFact')
bf = load_fnn('BuzzFeed')
all_df = pd.concat([pf, bf], ignore_index=True)

print(f'PolitiFact : {len(pf):,} rows  (fake={len(pf[pf.label=="fake"]):,}, real={len(pf[pf.label=="real"]):,})')
print(f'BuzzFeed   : {len(bf):,} rows  (fake={len(bf[bf.label=="fake"]):,}, real={len(bf[bf.label=="real"]):,})')
print(f'Combined   : {len(all_df):,} rows')
all_df.head(2)

## 1. Label Balance per Source

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, df) in zip(axes, [('PolitiFact', pf), ('BuzzFeed', bf)]):
    counts = df['label'].value_counts()
    ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
           colors=['#d73027', '#313695'], startangle=90)
    ax.set_title(f'{name} — label balance (n={len(df):,})')
plt.tight_layout()
plt.show()

## 2. Article Length Distribution

In [ ]:
all_df['char_len'] = all_df['text'].fillna('').str.len()
all_df['word_len'] = all_df['text'].fillna('').str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row_i, (source_name, sdf) in enumerate([('PolitiFact', pf), ('BuzzFeed', bf)]):
    sdf = sdf.copy()
    sdf['char_len'] = sdf['text'].fillna('').str.len()
    sdf['word_len'] = sdf['text'].fillna('').str.split().str.len()
    for col_i, (col, xlabel) in enumerate([('char_len', 'Characters'), ('word_len', 'Words')]):
        ax = axes[row_i][col_i]
        for label, color in [('fake', '#d73027'), ('real', '#313695')]:
            ax.hist(sdf[sdf.label == label][col].clip(upper=sdf[col].quantile(0.98)),
                    bins=40, alpha=0.6, label=label, color=color)
        ax.set_title(f'{source_name} — article length ({xlabel})')
        ax.set_xlabel(xlabel)
        ax.set_ylabel('Count')
        ax.legend()
plt.tight_layout()
plt.show()

print('Median word count by source and label:')
print(all_df.groupby(['source', 'label'])['word_len'].median().round(0))

## 3. Title Length Distribution

In [ ]:
all_df['title_word_len'] = all_df['title'].fillna('').str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, sdf) in zip(axes, [('PolitiFact', pf), ('BuzzFeed', bf)]):
    sdf = sdf.copy()
    sdf['twl'] = sdf['title'].fillna('').str.split().str.len()
    for label, color in [('fake', '#d73027'), ('real', '#313695')]:
        ax.hist(sdf[sdf.label == label]['twl'], bins=25, alpha=0.6, label=label, color=color)
    ax.set_title(f'{name} — headline word count')
    ax.set_xlabel('Words in headline')
    ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Missing Data

In [ ]:
for name, sdf in [('PolitiFact', pf), ('BuzzFeed', bf)]:
    missing = sdf.isnull().mean() * 100
    missing = missing[missing > 0].sort_values(ascending=False)
    print(f'\n{name} — % missing per column:')
    print(missing.round(1).to_string())

## 5. Publish Date Distribution

In [ ]:
all_df['publish_date'] = pd.to_datetime(all_df['publish_date'], errors='coerce')
valid_dates = all_df.dropna(subset=['publish_date'])
print(f'Records with valid publish_date: {len(valid_dates):,} / {len(all_df):,}')

if len(valid_dates) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    for label, color in [('fake', '#d73027'), ('real', '#313695')]:
        subset = valid_dates[valid_dates.label == label]
        ax.hist(subset['publish_date'], bins=40, alpha=0.6, label=label, color=color)
    ax.set_title('Publish date distribution')
    ax.set_xlabel('Date')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No valid dates to plot.')

## 6. Images availability

In [ ]:
def has_image(images_str):
    if pd.isna(images_str) or images_str in ('', '[]'):
        return False
    try:
        return len(ast.literal_eval(str(images_str))) > 0
    except Exception:
        return bool(images_str.strip())

all_df['has_image'] = all_df['images'].apply(has_image)

print('Articles with at least one image:')
print(all_df.groupby(['source', 'label'])['has_image'].mean().apply(lambda x: f'{x:.1%}'))

## 7. Source domain distribution

In [ ]:
from urllib.parse import urlparse

all_df['domain'] = all_df['url'].fillna('').apply(lambda u: urlparse(u).netloc or 'unknown')
top_domains = all_df.groupby(['domain', 'label']).size().unstack(fill_value=0)
top_domains['total'] = top_domains.sum(axis=1)
top_domains = top_domains.nlargest(20, 'total').drop(columns='total')

top_domains.plot(kind='barh', stacked=True, figsize=(9, 7),
                  color=['#d73027', '#313695'])
plt.title('Top 20 domains (fake vs real)')
plt.xlabel('Article count')
plt.gca().invert_yaxis()
plt.legend(title='label')
plt.tight_layout()
plt.show()

## 8. Benchmark readiness summary

| Property | PolitiFact | BuzzFeed |
|---|---|---|
| Total records | ~3,628 | ~1,932 |
| Label balance | ~50 / 50 | check above |
| Has body text | Yes (`text` column) | Yes |
| Has images | Partial | Partial |
| Has publish date | Partial | Partial |
| Pipeline input | `title + text[:500]` → `claim_text` | same |

**⚠️ Loader update needed:** The current `run_eval.py` loader expects the directory-tree format  
(`politifact/fake/<id>/news_content.json`). The Kaggle CSV format requires a new loader —  
see `TODO.md` item or update `benchmark/record.py` before running eval.

In [ ]:
print('PolitiFact columns:', list(pf.columns))
print('\nSample fake title:')
print(pf[pf.label=='fake']['title'].iloc[0])
print('\nSample real title:')
print(pf[pf.label=='real']['title'].iloc[0])